# Модуль 08 · Урок 1. Итоговый проект 1. Аналитика интернет-магазина

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## Данные для самостоятельного исследования

В архиве есть отдельный увеличенный CSV-набор этого проекта. Сначала пройдите задания на автопроверку, затем примените написанные функции к этим данным, проверьте агрегаты и объясните полученные результаты. Это не дополнительное оценочное задание: обязательные пять заданий проверяются автоматически ниже.

In [ ]:
# Данные для самостоятельного исследования после решения функций
project_orders = pd.read_csv(ROOT/'data'/'capstone_orders.csv')
project_clients = pd.read_csv(ROOT/'data'/'capstone_clients.csv')
print('Заказы:',project_orders.shape, '| Клиенты:',project_clients.shape)
print(project_orders.head(3)); print(project_clients.head(3))

## Сценарий проекта
Вы получили два набора: `orders` с заказами и `clients` со справочником клиентов. Руководству нужны ответы: сколько выручки принесли подтверждённые заказы, какие города и сегменты клиентов наиболее ценны, сколько неизвестных клиентов в транзакциях и как меняются продажи по месяцам. **Нельзя молча удалять записи с неизвестным клиентом**: это проблема качества данных, а не повод уменьшить выручку.

Задания расположены от сырых данных к итоговой витрине. Используйте функции из предыдущих модулей — фильтрацию, `assign`, `merge(validate=...)`, группировку, временные периоды. Тесты используют два разных магазина с разными ключами, индексами и датами.

## 1. Контракт финансовых показателей
Определим выручку по строке как `price * qty * (1 - discount/100)`. Отменённые заказы (`status == 'cancelled'`) исключаются из суммы; новые (`new`) НЕ считаются завершёнными продажами. Поэтому для отчётной выручки оставьте только `paid`. Количество заказов — количество строк, а не суммарное количество товаров. Средний чек — общая выручка / количество оплаченных заказов, а не среднее по всем строкам исходника.

In [ ]:
import pandas as pd
sample = pd.DataFrame({'status':['paid','new','paid'],'price':[100,120,80],'qty':[1,2,3],'discount':[0,10,0]})
paid = sample.loc[sample.status.eq('paid')].copy()
paid['revenue'] = paid.price * paid.qty * (1-paid.discount/100)
print('Выручка оплаченных:',paid.revenue.sum())

## 2. Достоверность JOIN
`clients` — справочник с уникальным `client_id`, поэтому его можно безопасно добавлять к заказам через `how='left'`, `validate='many_to_one'`. Пропущенный segment должен оставаться пропуском. Если объединили только paid-заказы, количество строк до и после join должно совпасть. На реальных данных дополнительная проверка `order_id` и кардинальности обязательна.

## 3. Временной анализ
Обрабатывайте даты через `pd.to_datetime()`. Для месячных метрик удобно `.dt.to_period('M')`. В итоговом месячном отчёте сохраняйте сортировку периодов. Выручку округлять во время расчётов нельзя: округление промежуточных значений иногда меняет итог.

## 4. Критерии сдачи
Пять решений должны выдавать правильные значения, колонки, индексы и порядок; входные таблицы должны оставаться неизменными. После прохождения автотестов объясните: какой знаменатель использовался для среднего чека, что произошло с неизвестными клиентами и почему новые заказы не включены в выручку.

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m8_01 · paid sales

Этап 1. Отберите оплаченные заказы; добавьте revenue после скидки; получите DataFrame `order_id`,`client_id`,`city`,`date`,`revenue`, индекс сбросьте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_01: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m8_01 = expected_for("m8_01", df=df)


In [ ]:
# m8_01: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 5 столбцов; колонки: `order_id`, `client_id`, `city`, `date`, `revenue`.

```text
  order_id  client_id    city        date  revenue
0  demo_o1          1  Москва  2026-01-03    254.0
1  demo_o3          1  Москва  2026-01-04    368.6
2  demo_o4          3  Казань  2026-02-01    317.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_01", result, _expected_m8_01)

### m8_02 · enrich sales

Этап 2. К подготовленным оплаченных продажам присоедините segment и region по client_id через LEFT merge с проверкой many_to_one.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `sales` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `date`, `revenue`.
- `client_ref` — `pd.DataFrame`, колонки: `client_id`, `segment`, `region`.


**Выполнение:** используйте готовые переменные `sales`, `client_ref`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_02: используются переменные sales, client_ref

sales = pd.DataFrame({
    'order_id': ['a', 'b'],
    'client_id': [1, 404],
    'city': ['A', 'B'],
    'date': ['2026-01-01', '2026-01-02'],
    'revenue': [20.0, 40.0],
})

client_ref = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})

print("sales — демонстрационные данные:")

display(sales)

print("client_ref — демонстрационные данные:")

display(client_ref)


# Снимок правильного результата для автопроверки.
_expected_m8_02 = expected_for("m8_02", sales=sales, client_ref=client_ref)


In [ ]:
# m8_02: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: sales, client_ref
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 7 столбцов; колонки: `order_id`, `client_id`, `city`, `date`, `revenue`, `segment`, `region`.

```text
  order_id  client_id city        date  revenue segment        region
0        a          1    A  2026-01-01     20.0     vip  Центр (демо)
1        b        404    B  2026-01-02     40.0     NaN           NaN
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_02", result, _expected_m8_02)

### m8_03 · city report

Этап 3. По таблице оплаченных продаж со столбцами city,revenue постройте DataFrame `city`,`orders`,`revenue`,`avg_check`, отсортированную по выручке убыванию, затем city по возрастанию; индекс сбросьте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `sales` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `revenue`.


**Выполнение:** используйте готовые переменные `sales`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_03: используются переменные sales

sales = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o3', 'demo_o4'],
    'client_id': [1, 1, 3],
    'city': ['Москва', 'Москва', 'Казань'],
    'category': ['чай', 'чай', 'книги'],
    'price': [127.0, 97.0, 317.0],
    'qty': [2, 4, 1],
    'discount': [0, 5, 0],
    'status': ['paid', 'paid', 'paid'],
    'date': ['2026-01-03', '2026-01-04', '2026-02-01'],
    'comment': ['  Быстрая доставка  ', 'Срочно', 'подарок'],
    'revenue': [254.0, 368.59999999999997, 317.0],
}, index=['z', 'a', 'd'])

print("sales — демонстрационные данные:")

display(sales)


# Снимок правильного результата для автопроверки.
_expected_m8_03 = expected_for("m8_03", sales=sales)


In [ ]:
# m8_03: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: sales
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 4 столбца; колонки: `city`, `orders`, `revenue`, `avg_check`.

```text
     city  orders  revenue  avg_check
0  Москва       2    622.6      311.3
1  Казань       1    317.0      317.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_03", result, _expected_m8_03)

### m8_04 · monthly sales

Этап 4. По оплаченной таблице с date,revenue постройте Series `revenue` по месяцам (Period[M]) в хронологическом порядке.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `sales` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `revenue`.


**Выполнение:** используйте готовые переменные `sales`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_04: используются переменные sales

sales = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o3', 'demo_o4'],
    'client_id': [1, 1, 3],
    'city': ['Москва', 'Москва', 'Казань'],
    'category': ['чай', 'чай', 'книги'],
    'price': [127.0, 97.0, 317.0],
    'qty': [2, 4, 1],
    'discount': [0, 5, 0],
    'status': ['paid', 'paid', 'paid'],
    'date': ['2026-01-03', '2026-01-04', '2026-02-01'],
    'comment': ['  Быстрая доставка  ', 'Срочно', 'подарок'],
    'revenue': [254.0, 368.59999999999997, 317.0],
}, index=['z', 'a', 'd'])

print("sales — демонстрационные данные:")

display(sales)


# Снимок правильного результата для автопроверки.
_expected_m8_04 = expected_for("m8_04", sales=sales)


In [ ]:
# m8_04: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: sales
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 2 значений; имя серии: `revenue`.

```text
month
2026-01    622.6
2026-02    317.0
Freq: M
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_04", result, _expected_m8_04)

### m8_05 · store kpis

Этап 5. Итоговые KPI из сырых заказов: dict `paid_orders`, `paid_revenue`, `avg_check` (для нуля оплаченных заказов 0.0), `cancelled_orders`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_05: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m8_05 = expected_for("m8_05", df=df)


In [ ]:
# m8_05: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'paid_orders': 3,
 'paid_revenue': 939.5999999999999,
 'avg_check': 313.2,
 'cancelled_orders': 1}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_05", result, _expected_m8_05)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()